In [ ]:
%pip install pylatexenc
%pip install matplotlib
# %pip install git+https://github.com/It4innovations/quantum-as-a-service.git@main

In [ ]:
import os

os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

# Connect to system

In [ ]:
from qheappe_client import QBackend, QJob, QProvider

lexis_project = "vlq_qaas_project"
resource_name = "VLQ-CZ-dev_qshceduler-alpha"

provider = QProvider(lexis_project)

## or define manually LEXIS access token e.g. via dotenv

## %pip install python-dotenv
## from dotenv import load_dotenv
## load_dotenv()

## provider = QProvider(LEXIS_PROJECT, token)

In [ ]:
# QBackend
backend: QBackend = provider.get_backend(
    resource_name,
    backend_name="VLQ-IQM-QCCSW-MOCK"
    )

print(f"Qubit: {backend.architecture.qubits}")

print(f"Gates: {backend.architecture.gates.keys()}")

# Define Quantum Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(15, 5)
for i in range(15):
    qc.h(i)
for i in range(14):
    qc.cx(i, i + 1)
qc.cx(0, 14)
qc.measure(range(5), range(5))

# Transpile Circuit

In [ ]:
# Transpile circuit Locally
qc_transpiled_local = backend.transpile(qc, optimize_single_qubits=False)

# Run Circuit

In [ ]:
from time import time

# Run circuit with number of SHOTS
SHOTS = 100

client_execution_time = time()

# Passing as argument not transpiled circuit
job: QJob = backend.run([qc_transpiled_local], shots=SHOTS)
result = job.result(qiskit_legacy=True)
client_execution_time = time() - client_execution_time
results_dict = result.get_counts()
print("Raw counts:", results_dict)

# Print non-zero results
for key, count in results_dict.items():
    if count > 0:
        print(f"State '{key}': {count} counts")
        # Only convert to int if key is actually a binary string
        if all(c in "01" for c in key):
            print(f"  -> Decimal value: {int(key, 2)}")

In [ ]:
job.job_id

In [ ]:
from qiskit.visualization import plot_histogram

# Plot histogram
plot_histogram(results_dict, figsize=(12, 6), bar_labels=False)  # hide text above bars